# Biomass gasification: how well do models transfer to *unseen studies*?

One notebook, run top to bottom (**Runtime > Run all**, about 15-25 minutes on free Colab). It needs internet access only to download the public data set from GitHub. Nothing is uploaded or tuned by hand.

**Data:** the open compilation of Ascher et al. (*Bioresource Technology* 2022, doi:10.1016/j.biortech.2022.128062): 312 gasification experiments from 29 publications (36 differently spelled study names in the file; the notebook merges spelling variants of the same paper). The notebook downloads the original file from the authors' GitHub repository (pinned to one commit) and does not redistribute it. Please cite that paper whenever you use these data.

**What this notebook does**
1. Predicts gas composition (H2, CO, CO2, CH4), the syngas share of the nitrogen-free gas (our "purity" measure), gas yield, gas heating value and cold-gas efficiency from feedstock properties and operating conditions.
2. Compares ordinary random cross-validation with **leave-one-study-out** validation (every experiment from one publication is withheld together).
3. Adds prediction intervals (conformal) and checks whether they really cover the truth for new studies.
4. Looks at the pressure subset and at feedstock groups, and ranks which inputs matter (permutation importance).

All outputs are saved in `gasification_results/` and zipped for download.

In [ ]:
import os, io, json, time, math, warnings, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import GroupKFold, KFold, LeaveOneGroupOut

SEED = 42
N_BOOT = 2000
OUT = "gasification_results"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
DATA_URL = ("https://raw.githubusercontent.com/SimonAscher/"
            "Interpretable-Machine-Learning-to-Model-Biomass-and-Waste-Gasification/"
            "b4b75058e64c799fddd0c334d68041440966c1ef/data/20220810_Dataset_Gasification_Ascher.xlsx")
SRC = os.environ.get("GAS_DATA") or DATA_URL
try:
    raw = pd.read_excel(SRC, sheet_name="Clean Data")
except Exception as e:
    print("Download failed:", repr(e)[:200])
    print("Please upload the file 20220810_Dataset_Gasification_Ascher.xlsx (from the authors' GitHub repository, folder data).")
    from google.colab import files
    up = files.upload()
    raw = pd.read_excel(io.BytesIO(next(iter(up.values()))), sheet_name="Clean Data")

need = ["ID","feed_type","C","H","N","S","O","feed_ash","feed_moisture","temperature","operating_pressure",
        "steam_biomass_ratio","ER","gasifying_agent","reactor_type","N2","H2","CO","CO2","CH4","gas_yield","gas_LHV","CGE","Reference"]
missing_cols = [c for c in need if c not in raw.columns]
assert not missing_cols, missing_cols
assert raw["Reference"].notna().all()
df = raw.copy()
import re
def canon_ref(s):
    # the same publication is spelled several ways in the file (trailing space, 'Bariateri', different end-page numbers)
    s = re.sub(r"\s+", " ", str(s)).strip().lower().replace("bariateri", "baratieri")
    return re.sub(r"(\d+)-\d+$", r"\1", s)
df["Reference_raw"] = df["Reference"]
df["Reference"] = df["Reference"].map(canon_ref)
merged = df.groupby("Reference")["Reference_raw"].nunique()
print("rows:", len(df), "| study names in file:", df["Reference_raw"].nunique(), "| publications after merging spelling variants:", df["Reference"].nunique())
print("merged publications:", {k: sorted(set(df.loc[df.Reference == k, "Reference_raw"])) for k in merged[merged > 1].index})
assert len(df) == 312 and df["Reference_raw"].nunique() == 36 and df["Reference"].nunique() == 29, "unexpected data set version"
df.head(3).T.head(12)

In [ ]:
# ---- clean and derive --------------------------------------------------------------------
num_cols = ["C","H","N","S","O","feed_LHV","feed_particle_size","feed_ash","feed_moisture","temperature",
            "residence_time","steam_biomass_ratio","ER"]
for c in num_cols + ["N2","H2","CO","CO2","CH4","C2Hn","gas_yield","gas_LHV","CGE"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# pressure: only a few experiments report a numeric pressure (kPa); the rest say 'atmospheric', 'slightly above', or are blank
p_num = pd.to_numeric(df["operating_pressure"], errors="coerce")
df["pressure_elevated"] = (p_num > 150).astype(float)       # 1 = clearly pressurised (>150 kPa), 0 = everything else
print("numeric pressures (kPa):", sorted(p_num.dropna().unique().tolist()),
      "| pressurised rows:", int(df["pressure_elevated"].sum()),
      "| studies with pressurised rows:", df.loc[df["pressure_elevated"]==1, "Reference"].nunique())

# targets (all vol.% on dry basis unless stated)
df["H2_CO"] = df["H2"] + df["CO"]
nofn2 = df["H2"] + df["CO"] + df["CO2"] + df["CH4"]
df["syngas_share_N2free"] = 100 * (df["H2"] + df["CO"]) / nofn2      # purity: H2+CO as % of H2+CO+CO2+CH4
TARGETS = {
    "H2": "H2 (vol.%)", "CO": "CO (vol.%)", "CO2": "CO2 (vol.%)", "CH4": "CH4 (vol.%)",
    "H2_CO": "H2+CO (vol.%)", "syngas_share_N2free": "H2+CO share of N2-free gas (%)",
    "gas_yield": "gas yield (Nm3/kg)", "gas_LHV": "gas LHV (MJ/Nm3)", "CGE": "cold-gas efficiency (%)"}
print(df[list(TARGETS)].describe().T[["count","mean","std","min","max"]].round(2))

n_bad = int((df["CGE"] > 100).sum())
print("cold-gas efficiency above 100% (physically impossible, data error or inconsistent basis):", n_bad, "rows -> excluded from the CGE target")
df.loc[df["CGE"] > 100, "CGE"] = np.nan
TARGETS["CGE"] = "cold-gas efficiency (%)"

# features: keep numeric columns with <=50% missing, plus categorical descriptors
NUM = [c for c in num_cols + ["pressure_elevated"] if c in df.columns and df[c].isna().mean() <= 0.5]
CAT = [c for c in ["feed_type","gasifying_agent","reactor_type","operating_condition","catalyst","scale","bed_material"] if c in df.columns]
for c in CAT:
    df[c] = df[c].astype("object").where(df[c].notna(), "missing").astype(str)
FEATS = NUM + CAT
dropped = [c for c in num_cols if c not in NUM]
print("numeric features used:", NUM)
print("categorical features used:", CAT)
print("numeric columns dropped for >50% missing:", dropped)
print("missing share of used numeric features:", df[NUM].isna().mean().round(2).to_dict())
groups_all = df["Reference"].astype(str).values

In [ ]:
# ---- models ------------------------------------------------------------------------------
class FeedTypeMean(BaseEstimator, RegressorMixin):
    """Baseline: predict the training mean of the same feedstock class."""
    def fit(self, X, y):
        s = pd.Series(np.asarray(y, float), index=X.index)
        self.g_ = s.groupby(X["feed_type"].astype(str)).mean()
        self.m_ = float(np.mean(y)); return self
    def predict(self, X):
        return X["feed_type"].astype(str).map(self.g_).fillna(self.m_).to_numpy(float)

def make_pre():
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5, sparse_output=False))])
    return ColumnTransformer([("num", num, NUM), ("cat", cat, CAT)])

MODELS = ["Mean", "FeedTypeMean", "Ridge", "RandomForest", "ExtraTrees", "HistGB"]
def make_model(name, n_est=200):
    if name == "Mean": return DummyRegressor(strategy="mean")
    if name == "FeedTypeMean": return FeedTypeMean()
    est = {"Ridge": Ridge(alpha=3.0),
           "RandomForest": RandomForestRegressor(n_estimators=n_est, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
           "ExtraTrees": ExtraTreesRegressor(n_estimators=n_est, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
           "HistGB": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=3, min_samples_leaf=10, random_state=SEED)}[name]
    return Pipeline([("pre", make_pre()), ("m", est)])

def loso_splits(g):
    return list(LeaveOneGroupOut().split(np.zeros(len(g)), groups=g))
def random_splits(n, rep):
    return list(KFold(5, shuffle=True, random_state=SEED + rep).split(np.arange(n)))
N_REP = 3

def oof_predict(name, X, y, splits, keep_models=False):
    pred = np.full(len(y), np.nan); models = []
    for tr, te in splits:
        m = make_model(name); m.fit(X.iloc[tr], y[tr]); pred[te] = m.predict(X.iloc[te])
        if keep_models: models.append((m, te))
    assert not np.isnan(pred).any()
    return (pred, models) if keep_models else pred

In [ ]:
# ---- main experiment: random vs leave-one-study-out -------------------------------------
store = {}      # (target, model, protocol) -> dict(abs_err, pred, r2, rmse)
loso_models = {}  # (target) -> list of (fitted RandomForest, test indices)  (for importance)
rows_idx = {}
def r2(y, p):
    return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

for t in TARGETS:
    ok = df[t].notna().values
    X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float); g = groups_all[ok]
    rows_idx[t] = np.where(ok)[0]
    sp_loso = loso_splits(g)
    sp_rand = [random_splits(len(y), r) for r in range(N_REP)]
    t1 = time.time()
    for mname in MODELS:
        keep = (mname == "RandomForest")
        res = oof_predict(mname, X, y, sp_loso, keep_models=keep)
        if keep: p, loso_models[t] = res
        else: p = res
        store[(t, mname, "loso")] = dict(abs_err=np.abs(y - p), pred=p, r2=r2(y, p), rmse=float(np.sqrt(np.mean((y - p) ** 2))))
        prs = [oof_predict(mname, X, y, sp_rand[r]) for r in range(N_REP)]
        store[(t, mname, "random")] = dict(abs_err=np.mean([np.abs(y - q) for q in prs], axis=0), pred=prs[0],
                                           r2=float(np.mean([r2(y, q) for q in prs])),
                                           rmse=float(np.mean([np.sqrt(np.mean((y - q) ** 2)) for q in prs])))
    mae = {m: store[(t, m, 'loso')]['abs_err'].mean() for m in ("Mean", "RandomForest")}
    print(f"{t:20s} n={len(y):3d} studies={len(set(g)):2d}  LOSO MAE mean-baseline={mae['Mean']:.3f}  RF={mae['RandomForest']:.3f}  ({time.time()-t1:.0f}s)")
print("elapsed min:", round((time.time() - T0) / 60, 1))

In [ ]:
# ---- tables with study-clustered bootstrap CIs --------------------------------------------
def boot_weights(g, seed):
    codes, uniq = pd.factorize(g); G = len(uniq)
    rng = np.random.default_rng(seed)
    return codes, G, rng.multinomial(G, np.ones(G) / G, size=N_BOOT).astype(float)

def boot_mae(a, codes, G, W):
    s = np.bincount(codes, weights=a, minlength=G); c = np.bincount(codes, minlength=G).astype(float)
    return (W @ s) / (W @ c)

def ci(x): return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

rows = []; gap_rows = []
for k, t in enumerate(TARGETS):
    g = groups_all[rows_idx[t]]; codes, G, W = boot_weights(g, SEED + k)
    bm = {(m, pr): boot_mae(store[(t, m, pr)]["abs_err"], codes, G, W) for m in MODELS for pr in ("loso", "random")}
    for m in MODELS:
        for pr in ("loso", "random"):
            d = store[(t, m, pr)]; mae = d["abs_err"].mean(); lo, hi = ci(bm[(m, pr)])
            sk = 1 - bm[(m, pr)] / bm[("Mean", pr)]; sk0 = 1 - mae / store[(t, "Mean", pr)]["abs_err"].mean()
            slo, shi = ci(sk)
            rows.append(dict(target=t, label=TARGETS[t], model=m, protocol=pr, n=len(g), n_studies=G, MAE=mae, MAE_lo=lo, MAE_hi=hi,
                             RMSE=d["rmse"], R2=d["r2"], skill_vs_mean=sk0, skill_lo=slo, skill_hi=shi))
    for m in ("RandomForest", "HistGB", "ExtraTrees", "Ridge"):
        diff = bm[(m, "loso")] - bm[(m, "random")]; dlo, dhi = ci(diff)
        gap_rows.append(dict(target=t, model=m, MAE_random=store[(t, m, "random")]["abs_err"].mean(),
                             MAE_loso=store[(t, m, "loso")]["abs_err"].mean(),
                             diff=store[(t, m, "loso")]["abs_err"].mean() - store[(t, m, "random")]["abs_err"].mean(),
                             diff_lo=dlo, diff_hi=dhi))
main = pd.DataFrame(rows); gap = pd.DataFrame(gap_rows)
main.to_csv(f"{OUT}/main_results_all_models.csv", index=False); gap.to_csv(f"{OUT}/random_vs_study_gap.csv", index=False)
show = main[(main.model == "RandomForest")].pivot(index="label", columns="protocol", values=["MAE", "R2", "skill_vs_mean"]).round(3)
print(show.to_string())
print()
print(gap[gap.model == "RandomForest"].round(3).to_string(index=False))

In [ ]:
# ---- which model is best under study-held-out validation? ---------------------------------
best = (main[main.protocol == "loso"].pivot(index="target", columns="model", values="MAE")[MODELS]).round(3)
best["best_model"] = best.drop(columns=["Mean", "FeedTypeMean"]).idxmin(axis=1)
best.to_csv(f"{OUT}/loso_mae_by_model.csv")
print(best.to_string())
print("\nWins (lowest LOSO MAE among learned models):", best["best_model"].value_counts().to_dict())

In [ ]:
# ---- conformal prediction intervals that are checked on unseen studies ---------------------
# For each held-out study, the interval width is calibrated ONLY from training studies (inner grouped CV),
# so the held-out study never influences its own interval.
CONF_T = ["H2", "CO", "H2_CO", "syngas_share_N2free", "gas_yield", "CGE"]
ALPHAS = (0.2, 0.1)
conf_rows = []; conf_study = []
for t in CONF_T:
    ok = rows_idx[t]; X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float); g = groups_all[ok]
    pred = store[(t, "RandomForest", "loso")]["pred"]
    cover = {a: np.zeros(len(y), bool) for a in ALPHAS}; width = {a: np.zeros(len(y)) for a in ALPHAS}
    for tr, te in loso_splits(g):
        gk = GroupKFold(n_splits=5); res = []
        for itr, ite in gk.split(np.zeros(len(tr)), groups=g[tr]):
            m = make_model("RandomForest", n_est=100); m.fit(X.iloc[tr[itr]], y[tr[itr]])
            res.append(np.abs(y[tr[ite]] - m.predict(X.iloc[tr[ite]])))
        res = np.concatenate(res); n = len(res)
        for a in ALPHAS:
            q = np.quantile(res, min(1.0, math.ceil((n + 1) * (1 - a)) / n), method="higher")
            cover[a][te] = np.abs(y[te] - pred[te]) <= q; width[a][te] = 2 * q
    for a in ALPHAS:
        sc = pd.DataFrame({"g": g, "c": cover[a]}).groupby("g")["c"].mean()
        conf_rows.append(dict(target=t, nominal=1 - a, coverage_pooled=cover[a].mean(), coverage_study_median=sc.median(),
                              coverage_study_min=sc.min(), share_studies_below_nominal=(sc < 1 - a).mean(),
                              mean_width=width[a].mean(), width_over_target_sd=width[a].mean() / y.std()))
        for gg, v in sc.items(): conf_study.append(dict(target=t, nominal=1 - a, study=gg, coverage=v))
    print(t, "done", round((time.time() - T0) / 60, 1), "min")
conf = pd.DataFrame(conf_rows); conf_s = pd.DataFrame(conf_study)
conf.to_csv(f"{OUT}/conformal_coverage.csv", index=False); conf_s.to_csv(f"{OUT}/conformal_coverage_by_study.csv", index=False)
print(conf.round(3).to_string(index=False))

In [ ]:
# ---- pressure subset and feedstock groups -------------------------------------------------
PT = ["H2", "CO", "gas_yield", "CGE"]
pr_rows = []
for t in PT:
    ok = rows_idx[t]; el = df.loc[ok, "pressure_elevated"].to_numpy() == 1
    for lab, mask in (("pressurised (>150 kPa)", el), ("all other", ~el)):
        if mask.sum() == 0: continue
        rf = store[(t, "RandomForest", "loso")]["abs_err"][mask]; mn = store[(t, "Mean", "loso")]["abs_err"][mask]
        pr_rows.append(dict(target=t, subset=lab, n=int(mask.sum()), n_studies=len(set(groups_all[ok][mask])),
                            MAE_RF=rf.mean(), MAE_mean_baseline=mn.mean(), skill=1 - rf.mean() / mn.mean()))
pressure = pd.DataFrame(pr_rows); pressure.to_csv(f"{OUT}/pressure_subset.csv", index=False)
print(pressure.round(3).to_string(index=False))
pressurised_studies = df.loc[df.pressure_elevated == 1, "Reference"].unique().tolist()
print("\nStudies with pressurised runs:", pressurised_studies)

ft_rows = []
for t in ["H2", "CO", "H2_CO", "gas_yield", "CGE"]:
    ok = rows_idx[t]; ft = df.loc[ok, "feed_type"].to_numpy()
    for f in sorted(set(ft)):
        mask = ft == f
        rf = store[(t, "RandomForest", "loso")]["abs_err"][mask]; mn = store[(t, "Mean", "loso")]["abs_err"][mask]
        ft_rows.append(dict(target=t, feed_type=f, n=int(mask.sum()), n_studies=len(set(groups_all[ok][mask])),
                            MAE_RF=rf.mean(), MAE_mean_baseline=mn.mean(), skill=1 - rf.mean() / mn.mean()))
feed = pd.DataFrame(ft_rows); feed.to_csv(f"{OUT}/error_by_feedstock.csv", index=False)
print(); print(feed.round(3).to_string(index=False))

In [ ]:
# ---- permutation importance on held-out studies (RandomForest) -----------------------------
IMP_T = ["H2", "CO", "gas_yield", "CGE"]
rng = np.random.default_rng(SEED); N_PERM = 3; imp_rows = []
for t in IMP_T:
    ok = rows_idx[t]; X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float)
    base = 0.0; perm = {f: 0.0 for f in FEATS}
    for m, te in loso_models[t]:
        m.set_params(m__n_jobs=1)          # tiny predictions: threads only add overhead
        Xt = X.iloc[te].copy(); yt = y[te]
        base += np.sum((yt - m.predict(Xt)) ** 2)
        for f in FEATS:
            acc = 0.0
            for _ in range(N_PERM):
                Xp = Xt.copy(); Xp[f] = rng.permutation(Xp[f].values)
                acc += np.sum((yt - m.predict(Xp)) ** 2)
            perm[f] += acc / N_PERM
    for f in FEATS:
        imp_rows.append(dict(target=t, feature=f, importance=(perm[f] - base) / base))
imp = pd.DataFrame(imp_rows); imp.to_csv(f"{OUT}/permutation_importance.csv", index=False)
for t in IMP_T:
    print(t, imp[imp.target == t].sort_values("importance", ascending=False).head(5)[["feature", "importance"]].round(3).values.tolist())

In [ ]:
# ---- figures --------------------------------------------------------------------------------
plt.rcParams.update({"font.size": 9})
rf = main[main.model == "RandomForest"]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
order = list(TARGETS)
y0 = np.arange(len(order))
r_ = rf[rf.protocol == "random"].set_index("target").loc[order]; l_ = rf[rf.protocol == "loso"].set_index("target").loc[order]
ax.barh(y0 - 0.2, r_["skill_vs_mean"], 0.38, label="random 5-fold", color="#9ecae1")
ax.barh(y0 + 0.2, l_["skill_vs_mean"], 0.38, label="leave-one-study-out", color="#08519c",
        xerr=[l_["skill_vs_mean"] - l_["skill_lo"], l_["skill_hi"] - l_["skill_vs_mean"]], capsize=2)
ax.set_yticks(y0); ax.set_yticklabels([TARGETS[t] for t in order]); ax.invert_yaxis(); ax.axvline(0, color="k", lw=0.8)
ax.set_xlabel("skill = 1 - MAE / MAE of predicting the training mean"); ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_skill_random_vs_study.png", dpi=200); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(9, 5.6))
cols = {"woody biomass": "#8c510a", "herbaceous biomass": "#1b7837", "municipal solid waste": "#762a83",
        "sewage sludge": "#636363", "plastics": "#d6604d", "other": "#4393c3"}
for ax, t in zip(axes.ravel(), ["H2", "CO", "syngas_share_N2free", "gas_yield", "gas_LHV", "CGE"]):
    ok = rows_idx[t]; y = df.loc[ok, t].to_numpy(float); p = store[(t, "RandomForest", "loso")]["pred"]; ft = df.loc[ok, "feed_type"].to_numpy()
    for f, c in cols.items():
        mk = ft == f
        if mk.any(): ax.scatter(y[mk], p[mk], s=9, color=c, alpha=0.75, label=f if t == "H2" else None)
    lim = [min(y.min(), p.min()), max(y.max(), p.max())]; ax.plot(lim, lim, "k--", lw=0.8)
    ax.set_title(f"{TARGETS[t]}\nMAE {np.abs(y-p).mean():.2f}, R2 {r2(y,p):.2f}", fontsize=8); ax.set_xlabel("measured"); ax.set_ylabel("predicted (held-out study)")
h_, l_ = axes[0, 0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=6, fontsize=7, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.savefig(f"{OUT}/fig2_parity_study_held_out.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.6))
d90 = conf_s[np.isclose(conf_s.nominal, 0.9)]; d80 = conf_s[np.isclose(conf_s.nominal, 0.8)]
pos = np.arange(len(CONF_T))
ax.boxplot([d80[d80.target == t].coverage for t in CONF_T], positions=pos - 0.18, widths=0.3, patch_artist=True,
           boxprops=dict(facecolor="#bdd7e7"), medianprops=dict(color="k"))
ax.boxplot([d90[d90.target == t].coverage for t in CONF_T], positions=pos + 0.18, widths=0.3, patch_artist=True,
           boxprops=dict(facecolor="#6baed6"), medianprops=dict(color="k"))
ax.axhline(0.8, color="#3182bd", ls="--", lw=0.8); ax.axhline(0.9, color="#08519c", ls="--", lw=0.8)
ax.set_xticks(pos); ax.set_xticklabels(CONF_T, rotation=20); ax.set_ylabel("coverage per held-out study")
ax.set_title("Light: 80% intervals, dark: 90% intervals (dashed = nominal)", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_conformal_coverage_by_study.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(9, 3.2))
for ax, t in zip(axes, ["H2", "gas_yield", "CGE"]):
    d = imp[imp.target == t].sort_values("importance", ascending=False).head(7).iloc[::-1]
    ax.barh(d.feature, d.importance, color="#2c7fb8"); ax.set_title(TARGETS[t], fontsize=8); ax.set_xlabel("increase in MSE when shuffled")
plt.tight_layout(); plt.savefig(f"{OUT}/fig4_permutation_importance.png", dpi=200); plt.show()

In [ ]:
# ---- summary and download -----------------------------------------------------------------------
rfl = main[(main.model == "RandomForest") & (main.protocol == "loso")].set_index("target")
rfr = main[(main.model == "RandomForest") & (main.protocol == "random")].set_index("target")
summary = dict(
    data_url=DATA_URL, n_rows=int(len(df)), n_studies=int(df["Reference"].nunique()),
    feature_numeric=NUM, feature_categorical=CAT, numeric_dropped_for_missing=dropped,
    pressurised_rows=int(df.pressure_elevated.sum()), pressurised_studies=pressurised_studies,
    sklearn=sklearn.__version__, seed=SEED, bootstrap_resamples=N_BOOT, random_cv_repeats=N_REP,
    loso_RF={t: dict(MAE=float(rfl.loc[t, "MAE"]), R2=float(rfl.loc[t, "R2"]), skill=float(rfl.loc[t, "skill_vs_mean"])) for t in TARGETS},
    random_RF={t: dict(MAE=float(rfr.loc[t, "MAE"]), R2=float(rfr.loc[t, "R2"]), skill=float(rfr.loc[t, "skill_vs_mean"])) for t in TARGETS},
    minutes=round((time.time() - T0) / 60, 1))
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
zname = "gasification_results.zip"
with zipfile.ZipFile(zname, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)): z.write(os.path.join(OUT, fn), fn)
print("saved:", sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download(zname)
except Exception:
    print("Not in Colab: the zip is in the current folder:", zname)

**When it finishes:** a file `gasification_results.zip` downloads automatically (if the download is blocked, open the folder icon on the left and download it by hand). Send that zip back to build the manuscript from your own run.